# run_decoder: decode BAR-ATH and write the submission

Loads the CatBoost model saved by `run_modeling.ipynb` (no retraining) and computes BAR-ATH's probabilities. The
decoder then labels them with the setting in `decoder_choice.json`, and the notebook writes `output/FORZA INDO.csv`.
As a check it also scores the labels against the published 5 minutes of ground truth; nothing is chosen with it.

`possession/choose_decoder.ipynb` chose `emission_scale` and `min_possession_s` on external data only: the best mean
spellF1 over the two external validation matches, labelled by the model trained without them. `TUNE_DECODER = True`
reruns the same grid here.

## Load

In [ ]:
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parent   # the repository root: notebooks run from possession/
PIPELINE_DIR = REPO_ROOT / "possession"
ARTIFACT_DIR = PIPELINE_DIR / "artifacts"
OUTPUT_DIR = PIPELINE_DIR / "outputs"


def load_notebook_defs(filename: str, namespace: dict) -> None:
    loaded = namespace.setdefault("_LOADED_NOTEBOOKS", set())
    if filename in loaded:
        return
    loaded.add(filename)
    nb = json.loads((PIPELINE_DIR / filename).read_text(encoding="utf-8"))
    for i, cell in enumerate(nb["cells"]):
        if cell["cell_type"] == "code" and "defs" in cell.get("metadata", {}).get("tags", []):
            exec(compile("".join(cell["source"]), f"{filename}::cell{i}", "exec"), namespace)

In [ ]:
load_notebook_defs("helper_data.ipynb", globals())        # DFL + Metrica loading, degradation, feature sets (for decoder tuning)
load_notebook_defs("helper_decoder.ipynb", globals())     # HMM decoder, tuning, metrics, label priors
load_notebook_defs("helper_reporting.ipynb", globals())   # evaluation on any ground truth, submission file

## Configuration

In [ ]:
MODEL_NAME = "possession_v1"              # the bundle saved by run_modeling.ipynb
GT_PATH = REPO_ROOT / "evaluation" / "gt" / "BAR-ATH_possession_gt_public.csv"   # the published 5 minutes: a check, never a choice
CHUNK_S = 300.0
TUNE_DECODER = False                      # False -> use the decoder knobs of decoder_choice.json
CHOICE_PATH = PIPELINE_DIR / "decoder_choice.json"       # written by possession/choose_decoder.ipynb, committed
DECODER_CHOICE = json.loads(CHOICE_PATH.read_text(encoding="utf-8")) if CHOICE_PATH.is_file() else {
    "emission_scale": None, "min_possession_s": None}   # run possession/choose_decoder.ipynb first
EMISSION_SCALE = DECODER_CHOICE["emission_scale"]     # smoothing knob
MIN_POSSESSION_S = DECODER_CHOICE["min_possession_s"] # shortest kept possession run before merging
ENABLE_X = False
ALLOW_DOWNLOAD = True
N_AUG = 2
TRAIN_STRIDE = 5
FORCE_REBUILD = False
SUBMISSION_PATH = REPO_ROOT / "output" / "FORZA INDO.csv"
PROBS_CACHE = ARTIFACT_DIR / f"{MODEL_NAME}_target_probs.npz"

## Step 1: load the saved model
`load_bundle` restores the CatBoost classifier, its feature columns and temperature, the decoder priors and the
feature and preprocessing configs. This step trains nothing.

In [ ]:
bundle = load_bundle(MODEL_NAME)
clf_ext = bundle["clf"]
FEAT_CFG, PRE_CFG = bundle["feat_cfg"], bundle["pre_cfg"]
HMM_BASE = bundle["dec_cfg"]                          # decoder priors (possession/dead lengths, handover share) measured in run_modeling
info = bundle["meta"]["info"]
EXTERNAL_MATCHES = list(info["external_matches"])
VAL_MATCHES = list(info["val_matches"])
print(f"loaded '{MODEL_NAME}': {bundle['meta']['trees']} trees, temperature {bundle['meta']['temperature']:.2f}")
print(f"external matches: {EXTERNAL_MATCHES}")
print(f"validation matches: {VAL_MATCHES}")

## Step 2: BAR-ATH preprocessing, features and probabilities

In [ ]:
inputs = load_match_inputs(GT_PATH)
clean = load_or_build_clean(inputs.raw, PRE_CFG, force=FORCE_REBUILD)
T = clean.n_frames
graded = clean.half_id() > 0
gt = inputs.gt
gblocks = gt_blocks(gt, clean.half_blocks)
X_target = WindowFeatureBuilder().transform(SpatialFeatureExtractor(FEAT_CFG).tabular(clean), clean.half_blocks)
probs_target = clf_ext.predict_proba(X_target, rows=np.flatnonzero(graded))    # the loaded CatBoost labels the target
print(f"frames={T} | graded={int(graded.sum())} | ground-truth frames={int((gt != 'X').sum())}")

## Step 3: probabilities on the validation matches (only with `TUNE_DECODER = True`)
With tuning on, the loaded model labels the two validation matches (J03WR9 and METRICA3) and the decoder grid runs on
their probabilities. The default skips this step.

In [ ]:
if TUNE_DECODER:
    profile = DegradationProfile.from_target(inputs.raw)
    val = {}
    for k, mid in enumerate(VAL_MATCHES):
        Fw_, lab, blocks = external_features(mid, profile, 10_000 + k, PRE_CFG, FEAT_CFG, ALLOW_DOWNLOAD, verbose=False)
        val[mid] = ExternalSplit(Fw_, lab, blocks)
    val_logits = {m_: clf_ext.predict_logits(v.X) for m_, v in val.items()}
    val_probs = {m_: softmax(val_logits[m_] / clf_ext.temperature) for m_ in VAL_MATCHES}
    val_lbl = [ids_to_labels(val[m_].y) for m_ in VAL_MATCHES]
    val_blocks = [val[m_].blocks for m_ in VAL_MATCHES]
    y_val = np.concatenate([val[m_].y for m_ in VAL_MATCHES])
    print("external validation, per frame:", frame_scores(np.concatenate([val_probs[m_] for m_ in VAL_MATCHES]), y_val))
else:
    print("TUNE_DECODER=False - skipping external validation features (not needed for the fixed decoder)")

## Step 4: decoder settings and decoding
By default the decoder uses the priors from the training matches and `EMISSION_SCALE` / `MIN_POSSESSION_S` from
`decoder_choice.json`. With `TUNE_DECODER = True` it searches the grid on the external validation matches instead.

In [ ]:
if TUNE_DECODER:
    t0 = time.time()
    HMM_CFG, tuning = tune_decoder([val_probs[m_] for m_ in VAL_MATCHES], val_lbl, val_blocks, HMM_BASE)
    print(f"decoder grid ({time.time() - t0:.0f}s) - best first:")
    print(tuning.head(8).to_string(index=False))
else:
    HMM_CFG = HMMConfig(**{**HMM_BASE.__dict__, "emission_scale": EMISSION_SCALE, "min_possession_s": MIN_POSSESSION_S})
    tuning = None
HMM_CFG = HMMConfig(**{**HMM_CFG.__dict__, "enable_x": ENABLE_X})
print(f"decoder: emission_scale={HMM_CFG.emission_scale}, min_possession_s={HMM_CFG.min_possession_s}, "
      f"mean_possession_s={HMM_CFG.mean_possession_s:.1f}, handover_share={HMM_CFG.handover_share:.2f}")

In [ ]:
lab_target = HMMSequenceDecoder(HMM_CFG).decode_match(probs_target, clean.half_blocks, clean.known_dead_ranges).labels

## Step 5: check the decoded labels against the published 5 minutes

In [ ]:
res = evaluate_on_gt(lab_target, probs_target, gt, clean, name="decoded (HMM)", chunk_s=CHUNK_S, plots=False)

## Step 6: save the bundle with its decoder settings, write the submission

In [ ]:
save_bundle(MODEL_NAME, clf_ext, HMM_CFG, FEAT_CFG, PRE_CFG,
            info={**info, "decoder_tuned": True}, root=ARTIFACT_DIR / "models")
final = predict_bundle(load_bundle(MODEL_NAME, root=ARTIFACT_DIR / "models"), clean)                     # the saved model + tuned decoder, loaded from disk
assert np.array_equal(final["labels"], lab_target), "the saved bundle does not reproduce the decoded labels"
sub = write_submission(final["labels"], SUBMISSION_PATH)
print(f"bundle re-saved (decoder tuned) | submission written + validated: {len(sub):,} rows -> {SUBMISSION_PATH}")
print(possession_stats(final["labels"], clean.half_blocks))

## Step 7: plots of the decoded labels (confusion matrix, predicted against true counts)

In [ ]:
plot_prediction_summary(gt, lab_target, clean.half_blocks, probs=probs_target, ball_src=clean.ball_src,
                        chunk_s=0, title="Decoded (HMM) - BAR-ATH",
                        path=FIG_DIR / "run_decoder_prediction.png")